In [1]:

!pip -q install pandas numpy plotly


In [2]:
# lets upload our dataset and unzip it
from google.colab import files
import zipfile
import io
import pandas as pd
import numpy as np
import plotly.express as px

uploaded = files.upload()


Saving 서울시 상권분석서비스(추정매출-상권)_2025년.zip to 서울시 상권분석서비스(추정매출-상권)_2025년.zip


In [3]:
#lets see what are we working with!
def read_korean_csv(data):
    for encoding in ["cp949", "utf-8-sig", "euc-kr", "utf-8"]:
        try:
            return pd.read_csv(
                io.BytesIO(data),
                encoding=encoding,
                low_memory=False
            )
        except UnicodeDecodeError:
            continue

    raise ValueError("Could not decode the CSV.")

frames = []

for filename, content in uploaded.items():
    if filename.lower().endswith(".zip"):
        with zipfile.ZipFile(io.BytesIO(content)) as archive:
            csv_files = [
                name for name in archive.namelist()
                if name.lower().endswith(".csv")
                and not name.startswith("__MACOSX/")
            ]

            print("CSV files found:", csv_files)

            for name in csv_files:
                data = archive.read(name)
                table = read_korean_csv(data)
                table["source_file"] = name
                frames.append(table)

    elif filename.lower().endswith(".csv"):
        table = read_korean_csv(content)
        table["source_file"] = filename
        frames.append(table)

df = pd.concat(frames, ignore_index=True)

print("Dataset loaded successfully!")
print("Rows:", len(df))
print("Columns:", len(df.columns))

display(df.head())


CSV files found: ['서울시 상권분석서비스(추정매출-상권)_2025년.csv']
Dataset loaded successfully!
Rows: 85732
Columns: 56


,기준_년분기_코드,상권_구분_코드,상권_구분_코드_명,상권_코드,상권_코드_명,서비스_업종_코드,서비스_업종_코드_명,당월_매출_금액,당월_매출_건수,주중_매출_금액,...,시간대_건수~24_매출_건수,남성_매출_건수,여성_매출_건수,연령대_10_매출_건수,연령대_20_매출_건수,연령대_30_매출_건수,연령대_40_매출_건수,연령대_50_매출_건수,연령대_60_이상_매출_건수,source_file
0,20251,U,관광특구,3001491,이태원 관광특구,CS300015,가방,1569075360,7398,1271375387,...,35,2010,5330,0,539,752,990,1393,3666,서울시 상권분석서비스(추정매출-상권)_2025년.csv
1,20251,U,관광특구,3001491,이태원 관광특구,CS200030,피부관리실,27388312,328,23404838,...,22,149,179,0,74,55,106,71,22,서울시 상권분석서비스(추정매출-상권)_2025년.csv
2,20251,U,관광특구,3001491,이태원 관광특구,CS200037,노래방,94383079,3307,40525543,...,863,2169,1064,56,1195,1450,368,84,80,서울시 상권분석서비스(추정매출-상권)_2025년.csv
3,20251,U,관광특구,3001491,이태원 관광특구,CS300016,안경,627053943,2570,349173694,...,0,1263,1283,0,722,1133,357,224,110,서울시 상권분석서비스(추정매출-상권)_2025년.csv
4,20251,U,관광특구,3001491,이태원 관광특구,CS300043,전자상거래업,34806521,713,28171327,...,0,132,581,0,379,314,0,10,10,서울시 상권분석서비스(추정매출-상권)_2025년.csv


In [4]:
#lets inspect and see for missing values/dubs
# Inspect column names
print("Dataset columns:")
for column in df.columns:
    print(column)

# Check missing values
missing = (
    df.isna().mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
)

print("\nMissing values (%):")
display(missing.to_frame("missing_percent"))

# Check duplicate rows
print("Duplicate rows:", df.duplicated().sum())

# Standardize column-name whitespace
df.columns = [
    str(column).strip().replace(" ", "")
    for column in df.columns
]


Dataset columns:
기준_년분기_코드
상권_구분_코드
상권_구분_코드_명
상권_코드
상권_코드_명
서비스_업종_코드
서비스_업종_코드_명
당월_매출_금액
당월_매출_건수
주중_매출_금액
주말_매출_금액
월요일_매출_금액
화요일_매출_금액
수요일_매출_금액
목요일_매출_금액
금요일_매출_금액
토요일_매출_금액
일요일_매출_금액
시간대_00~06_매출_금액
시간대_06~11_매출_금액
시간대_11~14_매출_금액
시간대_14~17_매출_금액
시간대_17~21_매출_금액
시간대_21~24_매출_금액
남성_매출_금액
여성_매출_금액
연령대_10_매출_금액
연령대_20_매출_금액
연령대_30_매출_금액
연령대_40_매출_금액
연령대_50_매출_금액
연령대_60_이상_매출_금액
주중_매출_건수
주말_매출_건수
월요일_매출_건수
화요일_매출_건수
수요일_매출_건수
목요일_매출_건수
금요일_매출_건수
토요일_매출_건수
일요일_매출_건수
시간대_건수~06_매출_건수
시간대_건수~11_매출_건수
시간대_건수~14_매출_건수
시간대_건수~17_매출_건수
시간대_건수~21_매출_건수
시간대_건수~24_매출_건수
남성_매출_건수
여성_매출_건수
연령대_10_매출_건수
연령대_20_매출_건수
연령대_30_매출_건수
연령대_40_매출_건수
연령대_50_매출_건수
연령대_60_이상_매출_건수
source_file

Missing values (%):


,missing_percent
기준_년분기_코드,0.0
상권_구분_코드,0.0
상권_구분_코드_명,0.0
상권_코드,0.0
상권_코드_명,0.0
서비스_업종_코드,0.0
서비스_업종_코드_명,0.0
당월_매출_금액,0.0
당월_매출_건수,0.0
주중_매출_금액,0.0


Duplicate rows: 0


In [5]:
#lets create chart out of them
sales_col = "당월_매출_금액"
industry_col = "서비스_업종_코드_명"

print("Sales field found:", sales_col in df.columns)
print("Industry field found:", industry_col in df.columns)

if sales_col in df.columns and industry_col in df.columns:

    df[sales_col] = pd.to_numeric(
        df[sales_col].astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    )

    industry_sales = (
        df.groupby(industry_col)[sales_col]
        .sum()
        .sort_values(ascending=False)
        .head(15)
        .sort_values()
        .reset_index()
    )

    fig = px.bar(
        industry_sales,
        x=sales_col,
        y=industry_col,
        orientation="h",
        title="Top 15 Industries by Estimated Sales",
        labels={
            sales_col: "Estimated sales (KRW)",
            industry_col: "Industry"
        }
    )

    fig.show()

else:
    print("Check the exact column names from Step 5.")


Sales field found: True
Industry field found: True


In [6]:
#lets see our KPIs
import pandas as pd
import numpy as np
import plotly.express as px
from IPython.display import display, HTML

# Identify the fields in your dataset
sales_col = "당월_매출_금액"
industry_col = "서비스_업종_코드_명"
area_col = "상권_코드_명"
period_col = "기준_년분기_코드"

# Convert sales to numeric
df[sales_col] = pd.to_numeric(
    df[sales_col].astype(str).str.replace(",", "", regex=False),
    errors="coerce"
)

total_rows = len(df)
total_sales = df[sales_col].sum()
industry_count = df[industry_col].nunique()
area_count = df[area_col].nunique() if area_col in df.columns else 0

display(HTML(f"""
<div style="display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:12px;font-family:Arial">
  <div style="padding:20px;border:1px solid #ddd;border-radius:12px">
    <div style="color:#666">Records</div>
    <h2>{total_rows:,}</h2>
  </div>
  <div style="padding:20px;border:1px solid #ddd;border-radius:12px">
    <div style="color:#666">Summed estimated sales</div>
    <h2>₩{total_sales:,.0f}</h2>
  </div>
  <div style="padding:20px;border:1px solid #ddd;border-radius:12px">
    <div style="color:#666">Industry categories</div>
    <h2>{industry_count:,}</h2>
  </div>
  <div style="padding:20px;border:1px solid #ddd;border-radius:12px">
    <div style="color:#666">Commercial areas</div>
    <h2>{area_count:,}</h2>
  </div>
</div>
"""))

# Top 10 industries
industry_summary = (
    df.groupby(industry_col, dropna=True)[sales_col]
      .sum()
      .nlargest(10)
      .sort_values()
      .reset_index()
)

fig = px.bar(
    industry_summary,
    x=sales_col,
    y=industry_col,
    orientation="h",
    title="Top 10 Industries — Estimated Sales",
    labels={
        sales_col: "Estimated sales (KRW)",
        industry_col: "Industry"
    },
    template="plotly_white"
)
fig.show()

# Top 10 commercial areas
if area_col in df.columns:
    area_summary = (
        df.groupby(area_col, dropna=True)[sales_col]
          .sum()
          .nlargest(10)
          .sort_values()
          .reset_index()
    )

    fig = px.bar(
        area_summary,
        x=sales_col,
        y=area_col,
        orientation="h",
        title="Top 10 Commercial Areas — Estimated Sales",
        labels={
            sales_col: "Estimated sales (KRW)",
            area_col: "Commercial area"
        },
        template="plotly_white"
    )
    fig.show()


In [9]:
#lets see weekday vs weekend sale
import pandas as pd
import plotly.express as px

# Find weekday and weekend sales columns
weekday_col = "월요일_매출_금액"
weekend_col = "주말_매출_금액"

print("Matching columns:")
print([c for c in df.columns if "매출_금액" in c and ("요일" in c or "주말" in c)])

if weekday_col in df.columns and weekend_col in df.columns:
    comparison = pd.DataFrame({
        "Day type": ["Weekday", "Weekend"],
        "Estimated sales (KRW)": [
            pd.to_numeric(df[weekday_col], errors="coerce").sum(),
            pd.to_numeric(df[weekend_col], errors="coerce").sum()
        ]
    })

    fig = px.bar(
        comparison,
        x="Day type",
        y="Estimated sales (KRW)",
        title="Weekday vs Weekend Estimated Sales",
        text_auto=".3s",
        template="plotly_white"
    )
    fig.update_layout(yaxis_title="Estimated sales (KRW)")
    fig.show()
else:
    print("The exact column names differ. Use the matching columns printed above.")


Matching columns:
['주말_매출_금액', '월요일_매출_금액', '화요일_매출_금액', '수요일_매출_금액', '목요일_매출_금액', '금요일_매출_금액', '토요일_매출_금액', '일요일_매출_금액']


In [11]:
#lets see sales amount by the time of the day
# Find sales-by-time-period columns
time_cols = [
    c for c in df.columns
    if "시간대" in c and "매출_금액" in c
]

print("Time-of-day sales columns:")
print(time_cols)

if time_cols:
    time_sales = pd.DataFrame({
        "Time period": [c.replace("_매출_금액", "") for c in time_cols],
        "Estimated sales (KRW)": [
            pd.to_numeric(df[c], errors="coerce").sum()
            for c in time_cols
        ]
    }).sort_values("Estimated sales (KRW)", ascending=False)

    fig = px.bar(
        time_sales,
        x="Time period",
        y="Estimated sales (KRW)",
        title="Estimated Sales by Time of Day",
        text_auto=".3s",
        template="plotly_white"
    )
    fig.update_layout(xaxis_title="Time period", yaxis_title="Estimated sales (KRW)")
    fig.show()
else:
    print("No matching time-of-day columns found. We can inspect the column names next.")


Time-of-day sales columns:
['시간대_00~06_매출_금액', '시간대_06~11_매출_금액', '시간대_11~14_매출_금액', '시간대_14~17_매출_금액', '시간대_17~21_매출_금액', '시간대_21~24_매출_금액']


In [12]:

# Identify the columns used for analysis
industry_col = "서비스_업종_코드_명"
sales_col = "당월_매출_금액"

if industry_col in df.columns and sales_col in df.columns:
    analysis_df = df[[industry_col, sales_col]].copy()
    analysis_df[sales_col] = pd.to_numeric(
        analysis_df[sales_col], errors="coerce"
    )
    analysis_df = analysis_df.dropna(subset=[industry_col, sales_col])

    industry_sales = (
        analysis_df.groupby(industry_col)[sales_col]
        .sum()
        .sort_values(ascending=False)
    )

    total_sales = industry_sales.sum()
    top_industry = industry_sales.index[0]
    top_value = industry_sales.iloc[0]

    print("INSIGHTIQ AI — AUTOMATED BUSINESS INSIGHTS")
    print("=" * 48)
    print(f"Number of industries: {len(industry_sales):,}")
    print(f"Top industry by summed estimated sales: {top_industry}")
    print(f"Top industry's summed estimated sales: ₩{top_value:,.0f}")

    if total_sales > 0:
        print(f"Share of summed industry totals: {top_value / total_sales:.1%}")

    print("\nTop 10 industries:")
    print(industry_sales.head(10).to_string())

    print(
        "\nInterpretation: These figures summarize the rows in the loaded dataset. "
        "They represent estimated sales, not profit. If rows overlap across "
        "commercial areas or categories, summed totals should not be interpreted "
        "as unique citywide revenue without further validation."
    )
else:
    print("Please check these columns:")
    print(df.columns.tolist())


INSIGHTIQ AI — AUTOMATED BUSINESS INSIGHTS
Number of industries: 62
Top industry by summed estimated sales: 한식음식점
Top industry's summed estimated sales: ₩15,059,664,523,557
Share of summed industry totals: 16.3%

Top 10 industries:
서비스_업종_코드_명
한식음식점     15059664523557
일반의류       7731024375028
일반의원       6731450402293
의약품        5631963312246
청과상        4867960545900
수산물판매      4363484071649
슈퍼마켓       3440852425599
편의점        3251412205729
반찬가게       3230402230439
일반교습학원     2935634094594

Interpretation: These figures summarize the rows in the loaded dataset. They represent estimated sales, not profit. If rows overlap across commercial areas or categories, summed totals should not be interpreted as unique citywide revenue without further validation.


In [14]:
#lets add some filters
import pandas as pd
import plotly.express as px
from IPython.display import display
import ipywidgets as widgets

industry_col = "서비스_업종_코드_명"
sales_col = "당월_매출_금액"

dashboard_df = df[[industry_col, sales_col]].copy()
dashboard_df[sales_col] = pd.to_numeric(
    dashboard_df[sales_col], errors="coerce"
)
dashboard_df = dashboard_df.dropna(subset=[industry_col, sales_col])

industry_options = ["All industries"] + sorted(
    dashboard_df[industry_col].astype(str).unique().tolist()
)

industry_filter = widgets.Dropdown(
    options=industry_options,
    value="All industries",
    description="Industry:",
    layout=widgets.Layout(width="500px")
)

def update_dashboard(industry):
    if industry == "All industries":
        filtered = dashboard_df
    else:
        filtered = dashboard_df[
            dashboard_df[industry_col].astype(str) == industry
        ]

    summary = (
        filtered.groupby(industry_col)[sales_col]
        .sum()
        .sort_values(ascending=False)
        .head(15)
        .reset_index()
    )

    fig = px.bar(
        summary,
        x=sales_col,
        y=industry_col,
        orientation="h",
        title=f"Estimated Sales by Industry — {industry}",
        labels={
            sales_col: "Estimated sales (KRW)",
            industry_col: "Industry"
        },
        template="plotly_white"
    )
    fig.update_layout(
        yaxis={"categoryorder": "total ascending"},
        height=650
    )
    fig.show()

widgets.interact(update_dashboard, industry=industry_filter);


interactive(children=(Dropdown(description='Industry:', layout=Layout(width='500px'), options=('All industries…

In [15]:
#lets add more grafics
area_col = "상권_코드_명"

if area_col in df.columns:
    area_df = df[[area_col, sales_col]].copy()
    area_df[sales_col] = pd.to_numeric(
        area_df[sales_col], errors="coerce"
    )
    area_df = area_df.dropna(subset=[area_col, sales_col])

    top_areas = (
        area_df.groupby(area_col)[sales_col]
        .sum()
        .sort_values(ascending=False)
        .head(15)
        .reset_index()
    )

    fig = px.bar(
        top_areas,
        x=sales_col,
        y=area_col,
        orientation="h",
        title="Top 15 Commercial Areas by Summed Estimated Sales",
        labels={
            sales_col: "Estimated sales (KRW)",
            area_col: "Commercial area"
        },
        template="plotly_white"
    )
    fig.update_layout(
        yaxis={"categoryorder": "total ascending"},
        height=650
    )
    fig.show()
else:
    print("Area column not found. Available columns:")
    print(df.columns.tolist())


In [16]:
#lets save
industry_summary = (
    dashboard_df.groupby(industry_col)[sales_col]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

industry_summary.to_csv(
    "insightiq_industry_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

area_summary = (
    df.groupby(area_col, dropna=False)[sales_col]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

area_summary.to_csv(
    "insightiq_area_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Export complete:")
print("- insightiq_industry_summary.csv")
print("- insightiq_area_summary.csv")


Export complete:
- insightiq_industry_summary.csv
- insightiq_area_summary.csv


In [17]:
#lets download the results
from google.colab import files

files.download("insightiq_industry_summary.csv")
files.download("insightiq_area_summary.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
#lets go on with building our dashboard

import pandas as pd
import plotly.express as px
from IPython.display import display, HTML

industry_col = "서비스_업종_코드_명"
sales_col = "당월_매출_금액"
area_col = "상권_코드_명"

work = df.copy()
work[sales_col] = pd.to_numeric(work[sales_col], errors="coerce")

industry_summary = (
    work.dropna(subset=[industry_col, sales_col])
    .groupby(industry_col)[sales_col]
    .sum()
    .sort_values(ascending=False)
)

area_summary = (
    work.dropna(subset=[area_col, sales_col])
    .groupby(area_col)[sales_col]
    .sum()
    .sort_values(ascending=False)
)

total_rows = len(work)
industry_count = work[industry_col].nunique()
area_count = work[area_col].nunique()
top_industry = industry_summary.index[0]
top_area = area_summary.index[0]

display(HTML(f"""
<div style="font-family:Arial;padding:20px;background:#101827;color:white;
border-radius:14px;">
  <h1 style="margin-bottom:5px;">InsightIQ AI</h1>
  <p style="color:#b9c5d8;">Seoul Commercial Intelligence | Executive Overview</p>
  <hr style="border-color:#344258;">
  <div style="display:flex;flex-wrap:wrap;gap:15px;">
    <div style="background:#1e293b;padding:18px;border-radius:10px;flex:1;min-width:160px;">
      <div style="color:#b9c5d8;">Dataset rows</div>
      <h2>{total_rows:,}</h2>
    </div>
    <div style="background:#1e293b;padding:18px;border-radius:10px;flex:1;min-width:160px;">
      <div style="color:#b9c5d8;">Industries</div>
      <h2>{industry_count:,}</h2>
    </div>
    <div style="background:#1e293b;padding:18px;border-radius:10px;flex:1;min-width:160px;">
      <div style="color:#b9c5d8;">Commercial areas</div>
      <h2>{area_count:,}</h2>
    </div>
  </div>
  <p style="margin-top:18px;">
    <b>Leading industry:</b> {top_industry}<br>
    <b>Leading area by summed estimates:</b> {top_area}
  </p>
  <p style="font-size:12px;color:#b9c5d8;">
    Sales are reported estimates, not profit. Summed values may include
    overlapping dimensions or geographic definitions.
  </p>
</div>
"""))

fig_industry = px.bar(
    industry_summary.head(10).sort_values(),
    orientation="h",
    title="Top 10 Industries by Summed Estimated Sales",
    labels={"value": "Estimated sales (KRW)", "index": "Industry"},
    template="plotly_white"
)
fig_industry.show()

fig_area = px.bar(
    area_summary.head(10).sort_values(),
    orientation="h",
    title="Top 10 Commercial Areas by Summed Estimated Sales",
    labels={"value": "Estimated sales (KRW)", "index": "Commercial area"},
    template="plotly_white"
)
fig_area.show()


In [19]:

report = f"""
INSIGHTIQ AI — SEOUL COMMERCIAL INTELLIGENCE
============================================

PROJECT OBJECTIVE
Analyze estimated sales patterns across Seoul commercial areas
and business categories.

DATASET OVERVIEW
- Rows analyzed: {len(df):,}
- Distinct industries: {df[industry_col].nunique():,}
- Distinct commercial areas: {df[area_col].nunique():,}

KEY FINDINGS
1. Leading industry by summed estimated sales: {top_industry}
   Estimated sales: KRW {industry_summary.iloc[0]:,.0f}

2. Leading commercial area by summed estimated sales: {top_area}
   Estimated sales: KRW {area_summary.iloc[0]:,.0f}

BUSINESS VALUE
The analysis can help analysts explore which business categories and
commercial areas merit further investigation.

LIMITATIONS
- Sales figures are estimates, not audited revenue or profit.
- Aggregated records may overlap across dimensions.
- Rankings describe this loaded dataset and its reporting period.
- Comparisons across years require checking geographic and methodology
  consistency first.

NEXT STEPS
- Validate aggregation rules against the official data dictionary.
- Investigate industry and area patterns in greater detail.
- Add demographic and time-of-day analysis.
"""

print(report)

with open("insightiq_findings.txt", "w", encoding="utf-8") as f:
    f.write(report)

print("\\nFindings report saved as insightiq_findings.txt")



INSIGHTIQ AI — SEOUL COMMERCIAL INTELLIGENCE

PROJECT OBJECTIVE
Analyze estimated sales patterns across Seoul commercial areas
and business categories.

DATASET OVERVIEW
- Rows analyzed: 85,732
- Distinct industries: 62
- Distinct commercial areas: 1,577

KEY FINDINGS
1. Leading industry by summed estimated sales: 한식음식점
   Estimated sales: KRW 15,059,664,523,557

2. Leading commercial area by summed estimated sales: 노량진역(노량진)
   Estimated sales: KRW 3,904,896,644,926

BUSINESS VALUE
The analysis can help analysts explore which business categories and
commercial areas merit further investigation.

LIMITATIONS
- Sales figures are estimates, not audited revenue or profit.
- Aggregated records may overlap across dimensions.
- Rankings describe this loaded dataset and its reporting period.
- Comparisons across years require checking geographic and methodology
  consistency first.

NEXT STEPS
- Validate aggregation rules against the official data dictionary.
- Investigate industry and area 